## 基于unstructured

> **输出目录**：本示例把 Markdown 写到 `../tmp/pdf-cleaning-demo/VMAX-S`，**不写 `../knowledge_md/`**。
> `knowledge_md/` 是课程语料，两篇综合实战课件（01 / 02）直接读取它做切分和检索，
> 一旦被本示例覆盖，检索结果会被静默改掉；`tmp/` 已在 `.gitignore` 中，随时可删可重跑。
>
> **依赖坑**：Unstructured 即使 `strategy="fast"` 也要用 spaCy 分句。
> 如果装到 `preshed==3.0.14`，`spacy.load()` 会抛 `OverflowError`，
> 而 Unstructured 内部把它 `except` 掉了，症状就是「Element数量: 0」。
> `uv sync --extra pdf` 已把 preshed 钉在 `<3.0.14`，装完请照上面那条命令重装。


> **本课件的实跑记录**：两个单元在 `env_rag_uv` 内核真实执行，合计约 37 秒（Unstructured 13 秒 / PyMuPDF 22 秒），0 报错。
> Unstructured 的 fast 策略在 5 份 PDF 上分别切出 `1037 / 741 / 1040 / 782 / 1013` 个 Element；
> PyMuPDF 路线 5 份全部成功；两条路线的 Markdown 都写到 `../tmp/pdf-cleaning-demo/VMAX-S`（合计约 4 300 行）。
> 前置：`cd ~/env_rag_uv && uv sync --extra pdf`（会拉 torch/timm/transformers/opencv，约 1.5 GB），
> 并且 preshed 必须 `<3.0.14`——否则 spaCy 加载失败被 Unstructured 吞掉，Element 数量会全部是 `0`。

In [ ]:
#!/usr/bin/env python
# coding: utf-8


from pathlib import Path
import re


from unstructured.partition.pdf import partition_pdf

from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Table,
    PageBreak,
    Header,
    Footer,
)



# =====================================================
# 路径配置
# =====================================================

INPUT_DIR = Path(
    "../knowledge_path/VMAX-S"
)


OUTPUT_DIR = Path(
    "../tmp/pdf-cleaning-demo/VMAX-S"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================================
# 文本清洗
# =====================================================

def clean_text(text: str):

    if not text:
        return ""


    # PDF特殊空格

    text = text.replace(
        "\u00a0",
        " "
    )

    text = text.replace(
        "\u3000",
        " "
    )


    # 连续空格

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )


    return text.strip()



# =====================================================
# Element 转 Markdown
# =====================================================

def elements_to_markdown(elements):

    md = []


    for element in elements:


        text = clean_text(
            element.text
        )


        if not text:

            continue



        # --------------------------
        # 标题
        # --------------------------

        if isinstance(
            element,
            Title
        ):


            md.append(
                "## " + text
            )

            md.append("")



        # --------------------------
        # 列表
        # --------------------------

        elif isinstance(
            element,
            ListItem
        ):


            md.append(
                "- " + text
            )



        # --------------------------
        # 表格
        # fast模式通常不会生成复杂表格
        # --------------------------

        elif isinstance(
            element,
            Table
        ):


            md.append(
                text
            )

            md.append("")



        # --------------------------
        # 正文
        # --------------------------

        elif isinstance(
            element,
            NarrativeText
        ):


            md.append(
                text
            )

            md.append("")



        # --------------------------
        # 页眉页脚
        # --------------------------

        elif isinstance(
            element,
            (
                Header,
                Footer
            )
        ):

            continue



        # --------------------------
        # 分页
        # --------------------------

        elif isinstance(
            element,
            PageBreak
        ):


            md.append(
                "\n---\n"
            )



        else:


            md.append(
                text
            )



    return "\n".join(md)



# =====================================================
# Markdown 后处理
# =====================================================

def clean_markdown(text):


    # 删除连续空行

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )


    return text.strip()



# =====================================================
# 单个 PDF
# =====================================================

def convert_pdf(pdf_file: Path):
    print("=" * 60)
    print(
        "处理:",
        pdf_file.name
    )
    
    # ==============================
    # Unstructured fast模式
    # 不下载layout模型
    # ==============================
    elements = partition_pdf(
        filename=str(pdf_file),

        # 关键：
        # 不调用YOLO layout模型
        strategy="fast",
        # 不启用布局表格模型
        infer_table_structure=False,
        # 保留分页
        include_page_breaks=True,

        languages=[
            "chi_sim",
            "eng"
        ]

    )

    print(
        "Element数量:",
        len(elements)
    )

    markdown = elements_to_markdown(
        elements
    )

    markdown = clean_markdown(
        markdown
    )


    output_file = (
        OUTPUT_DIR /
        pdf_file.with_suffix(".md").name
    )

    output_file.write_text(
        markdown,
        encoding="utf-8"
    )

    print(
        "生成:",
        output_file
    )



# =====================================================
# 批量处理
# =====================================================

pdf_files = sorted(
    INPUT_DIR.glob(
        "*.pdf"
    )
)


print(
    f"发现PDF数量: {len(pdf_files)}"
)



for pdf in pdf_files:
    try:
        convert_pdf(
            pdf
        )
    except Exception as e:
        print(
            "失败:",
            pdf.name
        )
        print(
            repr(e)
        )

print(
    "\n全部完成"
)

**预期输出：**

```text
~/env_rag_uv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
发现PDF数量: 5
============================================================
处理: ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf
Element数量: 1037
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.md
============================================================
处理: ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
Element数量: 741
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.md
============================================================
处理: ZXVMAX-S（V6.23）产品描述（数据业务）.pdf
Element数量: 1040
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.md
============================================================
处理: ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf
Element数量: 782
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.md
============================================================
处理: ZXVMAX-S（V6.23）产品描述（语音业务）.pdf
Element数量: 1013
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.md

全部完成
```

（输出里的 `~` 代表运行机器上的用户主目录，每台机器不同。）

## 基于文档定制化（不断调整）

In [ ]:
#!/usr/bin/env python
# coding: utf-8


import re
import fitz
from pathlib import Path



# =====================================================
# 路径配置
# =====================================================

INPUT_DIR = Path(
    "../knowledge_path/VMAX-S"
)


OUTPUT_DIR = Path(
    "../tmp/pdf-cleaning-demo/VMAX-S"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================================
# 参数配置
# =====================================================

# 如果封面目录不需要，可以设置
SKIP_PAGES = 0



# =====================================================
# 页眉页脚
# =====================================================


HEADER_PATTERNS = [

    r".*产品描述.*",

]


FOOTER_PATTERNS = [

    r"SJ-\d+.*",

    r"\d{4}-\d{2}-\d{2}",

]


PAGE_NUMBER_PATTERN = re.compile(
    r"^\d+$"
)



# =====================================================
# 基础清洗
# =====================================================

def clean_text(text):


    replace_map = {

        "\u00a0": " ",
        "\u3000": " ",

        # PDF项目符号
        "": "-",
        "": "-",
        "•": "-",
        "●": "-",
        "○": "-",
        "▪": "-",
        "▫": "-",
        "à": "-",

    }


    for k,v in replace_map.items():

        text=text.replace(
            k,
            v
        )


    text=re.sub(
        r"[ \t]+",
        " ",
        text
    )


    return text.strip()



# =====================================================
# 页眉页脚判断
# =====================================================

def is_header(text):


    for p in HEADER_PATTERNS:

        if re.match(
            p,
            text
        ):

            return True


    return False



def is_footer(text):


    for p in FOOTER_PATTERNS:

        if re.search(
            p,
            text
        ):

            return True


    return False



# =====================================================
# 标题识别
# =====================================================

def detect_heading(text):


    # 三级标题
    if re.match(
        r"^\d+\.\d+\.\d+\s+.+",
        text
    ):

        return "### " + text



    # 二级标题
    if re.match(
        r"^\d+\.\d+\s+.+",
        text
    ):

        return "## " + text



    # 一级标题
    if re.match(
        r"^\d+\s+.+",
        text
    ):


        # 防止：
        # 2 U
        # 35 kg

        if re.match(
            r"^\d+\s+(U|kg|mm|cm|GB|TB|GHz)$",
            text,
            re.I
        ):

            return None


        return "# " + text



    return None



# =====================================================
# 列表处理
# =====================================================

def normalize_bullet(text):


    if text.startswith("-"):

        value=text[1:].strip()

        return "- " + value



    return text



# =====================================================
# 判断是否需要合并换行
# =====================================================

def should_merge_line(
        previous,
        current,
        in_list=False
):


    if not previous:

        return False



    # 标题
    if current.startswith("#"):

        return False



    # 新列表
    if current.startswith("-"):

        return False



    # 当前属于列表正文

    if in_list:

        return True



    # 上一句没有结束

    if not previous.endswith(
        (
            "。",
            "！",
            "？",
            "；",
            "：",
            "）",
            "】",
            ".",
            ";",
            ":"
        )
    ):

        return True



    return False



# =====================================================
# 表格
# =====================================================

def extract_tables(page):


    tables=[]


    try:

        result=page.find_tables()


        for table in result.tables:

            tables.append(
                table.to_markdown()
            )


    except Exception:

        pass



    return tables



# =====================================================
# 页面解析
# =====================================================

def parse_page(
        page,
        page_no
):


    markdown=[]


    markdown.append(
        f"<!-- page_number:{page_no} -->"
    )


    markdown.append("")



    # ---------------------
    # 表格
    # ---------------------

    for table in extract_tables(page):

        markdown.append(
            table
        )

        markdown.append("")



    # ---------------------
    # 文本抽取
    # ---------------------

    lines=[]


    blocks=page.get_text(
        "dict"
    )["blocks"]



    for block in blocks:


        if "lines" not in block:

            continue



        for line in block["lines"]:


            text="".join(
                span["text"]
                for span in line["spans"]
            )


            text=clean_text(
                text
            )


            if not text:

                continue



            y=line["bbox"][1]



            # 页眉
            if y < 60:

                continue



            # 页脚
            if y > page.rect.height-60:

                continue



            if is_header(text):

                continue



            if is_footer(text):

                continue



            if PAGE_NUMBER_PATTERN.match(text):

                continue



            lines.append(
                text
            )



    # =================================================
    # 文本重组
    # =================================================

    result=[]


    buffer=""

    in_list=False


    pending_bullet=False



    for line in lines:



        # 单独项目符号

        if line=="-":

            pending_bullet=True

            continue



        if pending_bullet:

            line="- "+line

            pending_bullet=False



        # 标题

        heading=detect_heading(
            line
        )


        if heading:


            if buffer:

                result.append(
                    buffer
                )

                buffer=""


            result.append(
                heading
            )


            result.append("")

            in_list=False

            continue



        # 列表

        line=normalize_bullet(
            line
        )



        if line.startswith("-"):


            if buffer:

                result.append(
                    buffer
                )

                buffer=""


            result.append(
                line
            )


            in_list=True


            continue



        # 普通文本

        if should_merge_line(
            buffer,
            line,
            in_list
        ):

            buffer += line


        else:


            if buffer:

                result.append(
                    buffer
                )


            buffer=line


        in_list=False



    if buffer:

        result.append(
            buffer
        )



    return "\n".join(
        markdown + result
    )



# =====================================================
# Markdown清理
# =====================================================

def clean_markdown(text):


    # 多空行

    text=re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )


    return text.strip()



# =====================================================
# PDF转换
# =====================================================

def pdf_to_markdown(pdf_file):


    print(
        "处理:",
        pdf_file.name
    )


    pdf=fitz.open(
        pdf_file
    )


    pages=[]



    for page_no,page in enumerate(
        pdf,
        start=1
    ):


        if page_no <= SKIP_PAGES:

            continue



        pages.append(
            parse_page(
                page,
                page_no
            )
        )



    markdown="\n\n".join(
        pages
    )


    markdown=clean_markdown(
        markdown
    )



    output=(
        OUTPUT_DIR /
        pdf_file.with_suffix(".md").name
    )



    with open(
        output,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            markdown
        )


    print(
        "生成:",
        output
    )



# =====================================================
# 批量处理
# =====================================================

pdf_files=sorted(
    INPUT_DIR.glob(
        "*.pdf"
    )
)


print(
    "PDF数量:",
    len(pdf_files)
)



for pdf in pdf_files:


    try:

        pdf_to_markdown(
            pdf
        )


    except Exception as e:

        print(
            "失败:",
            pdf.name,
            e
        )



print(
    "全部完成"
)

**预期输出：**

```text
~/env_rag_uv/.venv/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
PDF数量: 5
处理: ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.md
处理: ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.md
处理: ZXVMAX-S（V6.23）产品描述（数据业务）.pdf
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.md
处理: ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.md
处理: ZXVMAX-S（V6.23）产品描述（语音业务）.pdf
生成: ../tmp/pdf-cleaning-demo/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.md
全部完成
```

（输出里的 `~` 代表运行机器上的用户主目录，每台机器不同。）